# 6-2절 연습 문제 풀이

이 노트북은 6-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch06/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 6장 연습에서 공통으로 사용하는 시 텍스트와 도구
POEM = '엄마야 누나야 강변 살자 뜰에는 반짝이는 금모래빛 뒷문밖에는 갈잎의 노래 엄마야 누나야 강변 살자'
EOS = '<eos>'

UNK = '<unk>'

def build_vocab(tokens):
    # 어휘 사전에 없는 토큰을 처리할 수 있도록 <unk>를 함께 넣는다.
    vocab = {t: i for i, t in enumerate([UNK] + sorted(set(tokens)))}
    return vocab, {i: t for t, i in vocab.items()}

def make_pairs(tokens, window=4):
    seq = list(tokens) + [EOS]
    return [(seq[i:i + window], seq[i + window])
            for i in range(len(seq) - window)]

# 6-2절 예제 - 작은별 멜로디와 RNN 모델
MELODY = '004455473322110744332217443322170044554733221107'
NOTE_NAMES = '도레미파솔라시_'

class MelodyRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=16, pick=-1):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.pick = pick        # 어떤 시점의 숨겨진 상태를 사용할지
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, self.pick, :])

def make_dataset(seq, vocab, window=4):
    idx = [vocab.get(t, vocab[UNK]) for t in seq]
    xs = torch.tensor([idx[i:i + window] for i in range(len(idx) - window)])
    ys = torch.tensor([idx[i + window] for i in range(len(idx) - window)])
    return nn.functional.one_hot(xs, len(vocab)).float(), ys

def train_rnn(model, X, Y, epochs=800, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

def generate(model, seed, vocab, rev, n=16, window=4):
    out = list(seed)
    for _ in range(n):
        idx = torch.tensor([[vocab.get(t, vocab[UNK]) for t in out[-window:]]])
        x = nn.functional.one_hot(idx, len(vocab)).float()
        out.append(rev[model(x).argmax(1).item()])
    return out[len(seed):]

## 연습 6-4

[연습 문제 6-1]에서 만든 어휘 사전을 사용해 역방향 어휘 사전, 데이터셋, 데이터로더를 만들어 보자. 각 글자를 토큰으로 하는 경우와 각 어절을 토큰으로 하는 경우 각각 크기가 5인 슬라이딩 윈도우를 사용한다.

In [ ]:
for name, tokens in [('글자', list(POEM)), ('어절', POEM.split())]:
    vocab, rev = build_vocab(tokens + [EOS])
    X, Y = make_dataset(tokens + [EOS], vocab, window=5)   # 슬라이딩 윈도우 5
    from torch.utils.data import TensorDataset, DataLoader
    loader = DataLoader(TensorDataset(X, Y), batch_size=4, shuffle=True)
    xb, yb = next(iter(loader))
    print(f'=== {name} 토큰 ===')
    print(f'  어휘 사전 {len(vocab)}개 / 역방향 사전 예: 0 -> {rev[0]!r}')
    print(f'  데이터셋 샘플 {len(X)}개, 입력 형태 {tuple(X.shape)}')
    print(f'  배치 하나: 입력 {tuple(xb.shape)}, 정답 {tuple(yb.shape)}')
    print()

역방향 어휘 사전은 모델이 출력한 인덱스를 다시 토큰으로 되돌릴 때 필요하다. 원-핫 입력의 형태는 (배치, 윈도우 크기, 어휘 사전 크기)가 된다.

## 연습 6-5

[연습 문제 6-4]에서 각 글자를 토큰으로 하는 데이터셋, 데이터로더, 어휘 사전과 역방향 어휘 사전을 사용해 MelodyRNN 모델을 학습한 후, '엄마야 ', '아빠야 ', '금강살자' 각각을 마중물 텍스트로 사용해 텍스트를 생성해 보고 다음 질문에 답해 보자.

모델의 성능은 어떠한가? 그렇게 평가한 이유와 함께 정리해 보자.

세 마중물 텍스트 중 예외가 발생한 경우가 있다면 그 이유는 무엇이며, 예외 처리 방법은 무엇일까?

In [ ]:
tokens = list(POEM) + [EOS]
vocab, rev = build_vocab(tokens)
X, Y = make_dataset(tokens, vocab, window=5)

torch.manual_seed(SEED)
model = MelodyRNN(len(vocab), hidden_size=32)
loss = train_rnn(model, X, Y, epochs=1500)
print(f'최종 손실 {loss:.4f}\n')

for seed in ['엄마야 ', '아빠야 ', '금강살자']:
    missing = [c for c in seed if c not in vocab]
    generated = generate(model, list(seed), vocab, rev, n=12, window=5)
    note = f'  <- 사전에 없는 글자 {missing}' if missing else ''
    print(f'{seed!r} -> {"".join(generated)}{note}')

**모델의 성능**: `'엄마야 '`처럼 학습 데이터에 그대로 등장한 마중물은 이어지는 구절을 잘 생성한다. `'금강살자'`는 글자는 모두 사전에 있지만 조합이 학습에 없어 엉뚱한 결과가 나온다.

**`'아빠야 '`는 아예 실행되지 않는다.** 시에 '아'와 '빠'가 없어 어휘 사전에도 없기 때문이다. 위 코드는 어휘 사전에 `<unk>` 토큰을 넣고 `vocab.get(t, vocab[UNK])`로 조회해 예외를 피했다. 이 처리가 없으면 `KeyError: '아'`가 발생한다.

**이유**: 학습 데이터가 시 한 편(약 50글자)뿐이라 모델이 **일반적인 한국어 규칙 대신 이 시 자체를 외운다**. 게다가 어휘 사전이 시에 등장한 글자로만 구성되어, 처음 보는 글자는 표현할 방법조차 없다.

생성 모델의 성능을 높이려면 데이터의 양과 다양성이 우선이며, 어휘 사전에는 미등록 토큰을 처리할 `<unk>` 같은 장치가 필요하다(9장에서 다시 다룬다).

## 연습 6-6

MelodyRNN 모델 정의에서 단순 RNN 계층의 출력을 [:, -1, :]으로 슬라이싱해 마지막 숨겨진 상태를 골라서 완전 연결 계층에 전달한다. 이때 [:, -2, :] 슬라이싱을 사용하면 모델이 학습하는 것은 무엇일지 유추해 보자.

In [ ]:
# 어느 시점의 숨겨진 상태를 쓰는지에 따라 무엇이 달라지는지 확인한다.
for pick in (-1, -2, -3):
    torch.manual_seed(SEED)
    model = MelodyRNN(len(vocab), hidden_size=32, pick=pick)
    loss = train_rnn(model, X, Y, epochs=1500)
    acc = (model(X).argmax(1) == Y).float().mean().item() * 100
    gen = generate(model, list('엄마야 '), vocab, rev, n=12, window=5)
    print(f'pick={pick}: 손실 {loss:.6f}, 학습 정확도 {acc:.1f}%, '
          f'생성 {"".join(gen)!r}')

`[:, -1, :]`은 **입력 5개를 모두 읽은 뒤의 상태**라 다음 토큰 예측에 필요한 정보를 전부 담고 있다.

`[:, -2, :]`를 쓰면 **마지막 입력 토큰을 보기 직전의 상태**로 예측하게 된다. 모델이 배우는 과제가 '5개를 보고 6번째를 맞히기'에서 사실상 **'앞의 4개만 보고 6번째를 맞히기'** 로 바뀌는 셈이다. 즉 한 칸 건너뛴 토큰을 예측하도록 학습된다.

**다만 실행 결과를 보면 -1과 -2의 성능 차이가 거의 없다**(손실 0.0278 수준으로 동일, 정확도 98%). 학습 데이터가 짧은 시 한 편이라 **앞의 4글자만으로도 시 안의 위치가 유일하게 특정되기** 때문이다.

차이는 정보를 더 버릴 때 드러난다. `pick=-3`으로 두면 손실이 약 두 배(0.0555)로 올라가고 정확도도 떨어진다. 즉 '마지막 상태를 쓴다'는 원칙은 옳지만, **그 효과의 크기는 데이터의 복잡도에 따라 달라진다**.

## 연습 6-7

단순 RNN 계층의 과거 기억 용량은 hidden_size로 결정되며, 이 값을 바꾸면 기억 크기와 파라미터 수가 함께 달라진다. 그러면 과거를 더 많이 기억한다고 생성 결과가 항상 좋아질까? 이 질문에 대한 답을 hidden_size를 4, 8, 16, 32, 64로 바꿔 가며 MelodyRNN 모델을 학습한 뒤, 학습 로그와 생성 결과를 비교해 찾아 보자.

In [ ]:
notes = list(MELODY)
note_vocab, note_rev = build_vocab(notes)
Xm, Ym = make_dataset(notes, note_vocab, window=4)

for hidden in (4, 8, 16, 32, 64):
    torch.manual_seed(SEED)
    model = MelodyRNN(len(note_vocab), hidden_size=hidden)
    loss = train_rnn(model, Xm, Ym, epochs=800)
    gen = generate(model, notes[:4], note_vocab, note_rev, n=12)
    n_param = sum(p.numel() for p in model.parameters())
    print(f'hidden={hidden:2d} (파라미터 {n_param:5,}개) 손실 {loss:.4f} '
          f'생성 {"".join(NOTE_NAMES[int(c)] for c in gen)}')

기억 용량을 키운다고 생성 결과가 계속 좋아지지는 않는다. `hidden_size`가 작으면 정보를 담지 못해 과소적합하지만, 지나치게 크면 **파라미터가 데이터 양에 비해 많아져 데이터를 외워 버린다**.

작은별 멜로디는 패턴이 단순해서 16~32 정도면 충분하고, 그 이상은 파라미터와 학습 시간만 늘어난다. 적정 용량은 **데이터의 복잡도**에 맞춰 정해야 한다.

## 연습 6-8

[도전 문제] MelodyRNN을 <반짝반짝 작은별>의 멜로디만으로 학습하는 대신 다음 세 동요의 멜로디를 추가해 네 동요의 음으로 학습 데이터를 구성해 모델을 만들어 본 후, 다음 질문에 답해 보자.

솔_미미솔미도_레_미레도미솔_도솔도솔도솔미_솔_레파미레도_ (<산토끼>)

솔솔라라솔솔미_솔솔미미레_솔솔라라솔솔미_솔미레미도_ (<학교종>)

솔미도레솔솔_솔미도레라솔_라라시_솔솔라_미라미도미레레_ (<노는 게 제일 좋아>의 앞부분)

<eos>가 정답인 샘플의 개수는 몇 개로 늘어날까?

한 곡만 학습한 모델과 비교해 생성된 멜로디의 자연스러움에는 어떤 변화가 있는가?

<eos> 토큰을 예측해 정해진 길이보다 짧게 생성하는 경우가 나타났는가? 만약 그렇지 않다면 학습 데이터를 어떤 방향으로 보강해야 할까?

In [ ]:
SONGS = {
    '작은별': MELODY,
    '산토끼': '솔_미미솔미도_레_미레도미솔_도솔도솔도솔미_솔_레파미레도_',
    '나비야': '솔솔라라솔솔미_솔솔미미레_솔솔라라솔솔미_솔미레미도_',
}
NAME_TO_IDX = {n: str(i) for i, n in enumerate('도레미파솔라시')}
NAME_TO_IDX['_'] = '7'

def to_digits(s):
    out, i = [], 0
    while i < len(s):
        if s[i] == '_':
            out.append('7'); i += 1
        else:
            out.append(NAME_TO_IDX[s[i]]); i += 1
    return out

all_notes = []
for name, song in SONGS.items():
    all_notes += (list(song) if song is MELODY else to_digits(song)) + ['7']
vocab4, rev4 = build_vocab(all_notes)
X4, Y4 = make_dataset(all_notes, vocab4, window=4)
print(f'네 곡 합친 토큰 {len(all_notes)}개, 샘플 {len(X4)}개')

torch.manual_seed(SEED)
model = MelodyRNN(len(vocab4), hidden_size=32)
print(f'최종 손실 {train_rnn(model, X4, Y4, epochs=1500):.4f}')
for name, song in SONGS.items():
    seed = (list(song) if song is MELODY else to_digits(song))[:4]
    gen = generate(model, seed, vocab4, rev4, n=12)
    print(f'{name}: {"".join(NOTE_NAMES[int(c)] for c in gen)}')

**여러 곡을 함께 학습하면 달라지는 점**

한 곡만 학습할 때는 다음 음이 거의 하나로 정해지지만, 여러 곡을 섞으면 **같은 4음 다음에 서로 다른 음이 오는 경우**가 생긴다. 모델은 이 충돌을 평균적인 확률 분포로 학습하게 되어, 특정 곡을 그대로 외우는 능력은 떨어지는 대신 '동요에서 자주 나오는 음의 흐름'이라는 일반적인 규칙을 배운다.

즉 **암기에서 일반화로** 성격이 바뀐다. 다만 곡을 구분할 단서(곡 ID 등)가 입력에 없어, 마중물만으로는 어떤 곡을 이어갈지 모호해지는 한계도 함께 나타난다.